# Generate your custom retrieval database

To use this notebook, run ```jupyter notebook``` from the notebook directory.
> ⚠️ **Warning:** You should run this command after setting up a development environment (-dev) as detailed  in the README file.

Install the necessary modules:

In [1]:
import os
import torch
from enum import Enum
from IPython.display import display, Markdown
from retriever import EmbeddingModel
from shared_utils.utils import get_leaf_classes
from rag_database_generator import ROOT_DIR
from rag_database_generator.utils import print_dict
from rag_database_generator.config import Config as RAGConfig
from rag_database_generator.embed import generate_embeddings
from rag_database_generator.parse import DoclingParser, ParsingOutputFormat
from rag_database_generator.chunk import generate_chunks, ChunkingStrategies

Set the config:

In [2]:
config = RAGConfig()

## Parse a PDF document

Set your parameters:

In [3]:
output_format = ".md"  # Available format are .md and .json

To parse the PDF, run:

In [4]:
parser = DoclingParser()

parsed_documents = parser.parse(files_to_keep=[os.path.join(ROOT_DIR, "tests", "data", "Medical.pdf")],
                               saving_folder=os.getcwd(),
                               output_format=ParsingOutputFormat(output_format))
content, name = parsed_documents[0]
parser = None
torch.cuda.empty_cache()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Saved: rag_database_generator/notebooks/Medical.md


Show the parsed document:

In [5]:
display(Markdown('>' + content.replace('\n', "\n>")))

>
>
>![AI generated alt text:  The image is a graphic with a red background and a white border. At the center, there is a white circle with a red drop shape inside it, containing what appears to be white crystals or particles. Below the circle, there is text in white that reads "2. Blood Glucose Monitoring." The style of the image is simple and informational, likely intended for educational or informational purposes.](image1.png)
>
>## Guidelines for blood glucose monitoring
>
>- · Always wash and dry your hands carefully before testing your blood glucose.
>- · A new lancet should be used for each test.
>- · Children should check their blood glucose under the supervision of a parent or an adult who understands what the readings mean.
>
>Blood glucose generally needs to be tested 5  -7 times every day before meals (breakfast, lunch and dinner) and before bed. It may be necessary to test more frequently at times, including during the night or when sick.
>
>## Blood glucose monitoring  what is it and why do we do it?
>
>Blood glucose monitoring means checking the blood glucose level to help keep it within the normal range (4-8 mmols/L). This is an essential part of looking after your child's diabetes. Blood glucose levels need to be monitored so that the insulin doses can be adjusted.
>
>## How do we do it and what other equipment is needed?
>
>- 1. We use a glucose meter: this measures blood glucose. You will be shown how to use the meter before you leave hospital.
>- 2. Finger -  lancing device: this is needed to prick the fi nger for the blood glucose test (it is essential to use a new lancet each time you test the blood glucose level).
>- 3. Test strips for the blood glucose and blood ketone meter: A drop of blood goes on the strip to measure blood glucose and/or blood ketone in the meter.
>
>
>
>![AI generated alt text:  The image is a stylized illustration depicting a hand holding a blood glucose meter with a reading of 5.9. The meter is connected to a test strip, which is being held by the hand. The test strip has a red and white label, and there are two additional test strips lying on the surface next to the meter. The background is plain white, emphasizing the subject matter.](image2.png)
>
>
>
>![AI generated alt text:  The image is a graphic with a yellow background and a white border. At the center, there is a stylized depiction of a bottle labeled "INSULIN" with a syringe and a dropper next to it. Below the bottle, there is text that reads "3. Using Insulin." The style of the image is simple and informative, likely intended for educational or instructional purposes.](image3.png)
>
>## How is the insulin given?
>
>There are three ways insulin can be given:
>
>- 1.  Insulin pen
>- 2.  A syringe and vial of insulin
>- 3.  A pump
>
>## Where do I inject the insulin?
>
>There are three main areas where insulin can be injected:
>
>## Using insulin to treat type 1 diabetes
>
>Insulin is the only way to manage type1 diabetes. Insulin is injected using a short needle into the tissue layer between the skin and the muscle (the subcutaneous tissue). This allows the insulin to be absorbed gradually.
>
>Everyone's lifestyle is different so we will work with you to fi nd the best regimen for your child. Over time you will learn to adjust the doses for different situations.
>
>Insulin pump therapy is another option for delivering insulin. It is rarely used at diagnosis but may be a suitable option as your child's diabetes journey progresses.
>
>![](image4.png)
>
>![](image5.png)
>
>![](image6.png)
>
>- 1. Abdomen    2.  Legs    3.  Buttock
>
>
>
>![AI generated alt text:  The image is a medical illustration showing a side view of a human body with a focus on the abdomen. It is divided into two sections, each labeled with numbers 1 through 3. The first section illustrates the positioning of the hands during a medical procedure, with the hands placed on the abdomen in a specific manner. The second section shows the same hands in a different position, indicating a change in the procedure. The illustration is educational, likely used to explain a medical technique or procedure.](image7.png)
>
>## Rotation &amp; care of injection sites
>
>- · Inspect and palpate (touch) injection site for lumps or bruising. If present, avoid injecting into that area until it has resolved.
>- · Rotate injection sites to prevent lumps from occurring. Correct rotation involves spacing insulin injections at least 1cm apart (approx. width of one adult fi nger) in the same injection zone. Your health care professional will advise you on this.
>- · Injecting through clothes is not a good idea as this may cause an infection at the site and the insulin may not get delivered into the subcutaneous layer and therefore may not work.
>- • Use a new needle for each injection.

## Chunk the parsed document

Available chunking strategies:

In [6]:
for cls_ in ChunkingStrategies:
    print(f" - {cls_.name}")

 - HIRAG
 - SPACY
 - NLTK
 - RECURSIVE
 - FIXED


Set your parameters:

In [7]:
chunking_method = "HIRAG"

To chunk the file, run:

In [8]:
chunks = generate_chunks(config=config,
                         files_to_keep=[os.path.join(os.getcwd(), "Medical.md")],
                         chunking_method=ChunkingStrategies[chunking_method],
                         saving_folder=os.getcwd())
torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Chunking text with HiRAG: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 57/57 [04:51<00:00,  5.11s/it]


Successfully saved rag_database_generator/notebooks/Medical_HiRAG_chunks.json


To show the 5 first generated chunks, run:

In [9]:
first_5_items = dict(list(chunks.items())[:5])
print_dict(first_5_items)

│  • 0: {'chunks': ['The background color is red.'], 'complete_chunks': 'What is the background color of the image?;The background color is red.', 'question': 'What is the background color of the image?', 'answer': 'The background color is red.'}
│  • 1: {'chunks': ['The background color is red.'], 'complete_chunks': 'What is the background color of the image?', 'question': 'What is the background color of the image?', 'answer': 'The background color is red.'}
│  • 2: {'chunks': ['The dominant hue is red.'], 'complete_chunks': 'What is the dominant hue of the image?;The dominant hue is red.', 'question': 'What is the dominant hue of the image?', 'answer': 'The dominant hue is red.'}
│  • 3: {'chunks': ['The dominant hue is red.'], 'complete_chunks': 'What is the dominant hue of the image?', 'question': 'What is the dominant hue of the image?', 'answer': 'The dominant hue is red.'}
│  • 4: {'chunks': ['The backdrop is red.'], 'complete_chunks': 'What color is the backdrop of the image?;

## Generate RAG database

Available embedding models:

In [10]:
for cls_ in get_leaf_classes(EmbeddingModel):
    print(f" - {cls_.name}")

 - all-MiniLM-L6-v2
 - GIST-all-MiniLM-L6-v2
 - gte-small-zh


Set your parameters:

In [11]:
embedding_model = "all-MiniLM-L6-v2"

To generate the database, run:

In [ ]:
file_names = ["Medical_HiRAG_chunks.json"]  # The file name(s) must correspond to the one generated in the chunking section

In [12]:
rag_database = generate_embeddings(config=config,
                                   files_to_keep=[os.path.join(os.getcwd(), name) for name in file_names],
                                   embedding_model=embedding_model,
                                   saving_folder=os.getcwd())

Successfully saved /rag/example_notebooks/rag_database.pkl
